# PDF RAG Pipeline — Step by Step

This notebook walks through a **Retrieval-Augmented Generation (RAG)** pipeline for PDF documents, one step at a time.

## What is RAG?

Large Language Models (LLMs) like GPT or Qwen don't know about your private documents. **RAG** fixes this by:

1. **Indexing** your documents — turning them into numerical vectors stored in a database.
2. **Retrieving** the chunks most relevant to a user's question.
3. **Generating** an answer by passing those chunks + the question to the LLM.

```
PDFs → split into chunks → embed as vectors → store in Chroma
                                                    ↓
                question → embed → similarity search → top chunks → LLM → answer
```

## Prerequisites
- `venv2` activated with all packages installed
- `HF_TOKEN` set in `.env`
- PDF files in `../data/pdf/`

## Step 0 — Imports & Environment

We load:
- **`dotenv`** — reads `.env` so `HF_TOKEN` is available to the HuggingFace client
- **LangChain pieces** — the building blocks we'll wire together

In [1]:
import os
import sys
from dotenv import load_dotenv

# Make the project root importable so we can reach config/
sys.path.append('..')

# Load HF_TOKEN and other env vars from ../.env
load_dotenv('../.env')

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
    ChatHuggingFace,
)
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA

print('Imports OK')
print('HF_TOKEN loaded:', bool(os.getenv('HF_TOKEN')))

d:\Linkedin\RAG\RAG-Project\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
HF_TOKEN loaded: True


## Step 1 — The Embedding Model

An **embedding model** converts text into a vector (a list of numbers, e.g. 384 floats). Pieces of text with similar meaning produce vectors that are close together in space.

We use `all-MiniLM-L6-v2` — small, fast, runs locally on CPU, no API calls.

In [2]:
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Quick sanity check — embed a sentence
sample_vector = embeddings.embed_query('Hello, how are you?')
print(f'Vector length: {len(sample_vector)}')
print(f'First 5 numbers: {sample_vector[:5]}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 877.05it/s]


Vector length: 384
First 5 numbers: [0.019096774980425835, 0.034465137869119644, 0.09162800759077072, 0.07016535103321075, -0.029946569353342056]


## Step 2 — The LLM (Language Model)

We use **Qwen 2.5 7B Instruct** via the HuggingFace Inference API. It runs in the cloud, not on your machine — that's why you need `HF_TOKEN`.

Two wrappers:
- `HuggingFaceEndpoint` — the raw HTTP client to HF
- `ChatHuggingFace` — wraps it to behave like a chat model (system/user/assistant messages)

In [3]:
endpoint = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task='conversational',
    temperature=0.7,        # 0 = deterministic, 1 = creative
    max_new_tokens=512,     # cap on answer length
)
llm = ChatHuggingFace(llm=endpoint)

# Test it with a plain question (no RAG yet)
response = llm.invoke('Say hello in one short sentence.')
print(response.content)

Hello!


## Step 3 — Load the PDFs

`PyPDFDirectoryLoader` walks a folder and loads every PDF it finds. Each PDF page becomes one LangChain `Document` with text + metadata (`source` = file path, `page` = page index).

In [4]:
pdf_dir = '../data/pdf'
loader = PyPDFDirectoryLoader(pdf_dir, recursive=True)
raw_documents = loader.load()

print(f'Loaded {len(raw_documents)} pages from PDFs in {pdf_dir}')
print('---')
print('First page preview:')
print('Source:', raw_documents[0].metadata.get('source'))
print('Page:', raw_documents[0].metadata.get('page'))
print('First 300 chars:')
print(raw_documents[0].page_content[:300])

Loaded 144 pages from PDFs in ../data/pdf
---
First page preview:
Source: ..\data\pdf\paper_1.pdf
Page: 0
First 300 chars:
1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai K


## Step 4 — Split into Chunks

**Why split?** A single PDF page can be thousands of characters. LLMs have token limits, and similarity search works better on focused, smaller pieces of text.

**`RecursiveCharacterTextSplitter`** breaks text along natural boundaries (paragraphs → sentences → words):
- `chunk_size=1000` — each chunk is at most ~1000 characters
- `chunk_overlap=200` — consecutive chunks share 200 chars so context isn't lost mid-sentence

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
documents = splitter.split_documents(raw_documents)

print(f'{len(raw_documents)} pages -> {len(documents)} chunks')
print('---')
print('Sample chunk:')
print(documents[0].page_content[:400])

144 pages -> 670 chunks
---
Sample chunk:
1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University
cCollege of Design and I


## Step 5 — Build the Vector Store

**Chroma** is a local vector database. We feed it:
1. The chunks
2. The embedding model (so Chroma knows how to turn chunks into vectors)
3. A directory to persist the index so we don't re-embed on every run

This step **embeds every chunk** — it takes a moment the first time.

In [6]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory='../data/chroma_db_pdf',
)
print(f'Vector store built with {vector_store._collection.count()} embedded chunks')

Vector store built with 1936 embedded chunks


## Step 6 — The Retriever

A **retriever** is a thin wrapper around the vector store that, given a question, returns the top-k most similar chunks.

We pick `k=4` — four chunks of context is usually enough without overwhelming the LLM.

In [7]:
retriever = vector_store.as_retriever(search_kwargs={'k': 4})

# Try retrieval directly (no LLM yet) to see what comes back
test_question = 'What were Dubai residential transaction volumes in Q1 2025?'
retrieved = retriever.invoke(test_question)

print(f'Retrieved {len(retrieved)} chunks for: {test_question!r}\n')
for i, doc in enumerate(retrieved, 1):
    src = os.path.basename(doc.metadata.get('source', '?'))
    page = doc.metadata.get('page')
    print(f'--- Chunk {i} | {src} (page {page + 1 if page is not None else "?"}) ---')
    print(doc.page_content[:250], '...\n')

Retrieved 4 chunks for: 'What were Dubai residential transaction volumes in Q1 2025?'

--- Chunk 1 | Dubai-Residential-Market-Performance-Q12025.pdf (page 2) ---
2 Dubai Residential Market Performance Q1 2025
The UAE’s GDP is forecasted to grow by 4.7% in 2025, 
while Dubai is expected to see a 3.3% increase. This 
sustained economic expansion, supported by 
population growth and continued investor conﬁdence, ...

--- Chunk 2 | Dubai-Residential-Market-Performance-Q12025.pdf (page 2) ---
2 Dubai Residential Market Performance Q1 2025
The UAE’s GDP is forecasted to grow by 4.7% in 2025, 
while Dubai is expected to see a 3.3% increase. This 
sustained economic expansion, supported by 
population growth and continued investor conﬁdence, ...

--- Chunk 3 | dubai-residential-market-report---q2-2025.pdf (page 1) ---
all contributing to a clear shift toward homeownership among 
expatriates and continued appeal for investors.
Q2 2025 broke the 50,000 threshold for transaction volumes 
marking 

## Step 7 — Build the QA Chain

`RetrievalQA` ties everything together:
- Takes a question
- Asks the retriever for relevant chunks
- Stuffs them into a prompt (`chain_type='stuff'`)
- Sends prompt + question to the LLM
- Returns the answer

`return_source_documents=True` makes the chain also return *which chunks it used*, so we can cite sources.

In [8]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
)
print('QA chain ready.')

QA chain ready.


## Step 8 — Ask Questions!

Helper function that asks a question and prints the answer + sources.

In [9]:
def ask(question: str):
    result = qa_chain.invoke({'query': question})
    print(f'Q: {question}\n')
    print(f'A: {result["result"]}\n')
    sources = set()
    for doc in result.get('source_documents', []):
        src = os.path.basename(doc.metadata.get('source', 'unknown'))
        page = doc.metadata.get('page')
        sources.add(f'{src}' + (f' (page {page + 1})' if page is not None else ''))
    if sources:
        print('Sources:')
        for s in sorted(sources):
            print(f'  - {s}')
    print()

ask('Summarize the Dubai residential market performance in Q1 2025.')

Q: Summarize the Dubai residential market performance in Q1 2025.

A: I don't have specific data on the Dubai residential market performance for Q1 2025 to summarize. The information you provided refers to "Dubai Residential Market Performance Q1 2025" and "Dubai Residential Market Performance FY 2025," but no actual performance details are given. To provide an accurate summary, I would need detailed metrics such as prices, sales volume, rental rates, or other relevant market indicators for Q1 2025.

Sources:
  - Dubai-Residential-Market-Performance-Q12025.pdf (page 1)
  - Dubai-Residential-Market-Performance_FY2025.pdf (page 1)



In [10]:
ask('Which areas had the most apartment completions?')

Q: Which areas had the most apartment completions?

A: The provided context does not specify which areas had the most apartment completions. It focuses on rental yields and market shares but does not give information about apartment completions by area. Therefore, I don't know the answer to which areas had the most apartment completions based on the given information.

Sources:
  - Dubai-Residential-Market-Performance-Q12025.pdf (page 6)
  - Dubai-Residential-Market-Performance_FY2025.pdf (page 18)



In [ ]:
ask('How did villa prices change year over year?')

Q: How did villa prices change year over year?

A: Based on the provided context, villa prices saw a significant change year over year from 2020 to 2024. Specifically, villa prices almost doubled during this period, increasing from an average of around AED 7,575 per square meter in 2020 to AED 14,605 per square meter in 2024. This represents a 93% surge in villa prices, reflecting the post-pandemic shift towards larger, private, low-density living spaces and the constrained supply of prime villa properties in key areas.

Sources:
  - annual_report_2024_english.pdf (page 19)
  - annual_report_2024_english.pdf (page 49)



: 

## Try Your Own

Replace the question below with anything you want to ask about the PDFs.

In [ ]:
ask('YOUR QUESTION HERE')

## Recap

| Step | What we did | Tool |
|---|---|---|
| 1 | Turn text into vectors | `HuggingFaceEmbeddings` |
| 2 | Connect to a chat LLM | `ChatHuggingFace` + `HuggingFaceEndpoint` |
| 3 | Load PDF pages | `PyPDFDirectoryLoader` |
| 4 | Split pages into chunks | `RecursiveCharacterTextSplitter` |
| 5 | Index chunks for fast lookup | `Chroma` |
| 6 | Find relevant chunks for a question | `as_retriever` |
| 7 | Chain retrieval + LLM | `RetrievalQA` |
| 8 | Ask questions | `qa_chain.invoke` |

## Where to go next
- Tweak `chunk_size` / `chunk_overlap` in Step 4 and see how answers change
- Try a different `k` in Step 6 (more chunks = more context, but slower & noisier)
- Swap the LLM in Step 2 for another HF chat model
- Add a custom prompt template to `RetrievalQA` to control answer style